In [ ]:
import requests # Import the requests library for making HTTP requests

from config import HF_API_KEY # Import the Hugging Face API key from a 'config' file (assumed to exist)


# Define the model ID for the Hugging Face model to be used for zero-shot classification
MODEL_ID = "facebook/bart-large-mnli"

# Construct the API URL for the specified Hugging Face model
API_URL = f"https://router.huggingface.co/hf-inference/models/{MODEL_ID}"

# Set the authorization headers, including the API key, for the HTTP requests
HEADERS = {"Authorization": f"Bearer {HF_API_KEY}"}

# Define a list of candidate topics that the model will classify news headlines into
TOPICS = ["Sports", "Technology", "Business", "Politics", "Health"]


# Define a function to send a headline to the Hugging Face API for classification
def ask_hf(headline: str):
    # Create the payload for the API request, including the headline and candidate labels
    payload = {"inputs": headline, "parameters": {"candidate_labels": TOPICS}}
    # Send a POST request to the Hugging Face API with the payload and headers
    r = requests.post(API_URL, headers=HEADERS, json=payload, timeout=30)
    # Check if the request was successful; if not, raise a RuntimeError
    if not r.ok:
        raise RuntimeError(f"HF error {r.status_code}: {r.text}")
    # Return the JSON response from the API, which contains classification predictions
    return r.json()  # returns a LIST of {"label": ..., "score": ...}


# Define a function to find the best topic and its score from the prediction list
def best_topic(preds: list):
    # Find the prediction with the highest score
    best = max(preds, key=lambda x: x["score"])
    # Return the label and score of the best topic
    return best["label"], best["score"]


# Define a function to create a confidence bar visualization
def bar(score: float) -> str:
    # Convert the score to a percentage
    pct = score * 100
    # Calculate the number of filled blocks for the bar (each block represents 10%)
    blocks = int(pct // 10)
    # Return a string representing the bar with filled and empty blocks
    return "█" * blocks + "░" * (10 - blocks)


# Define a function to display the classification results in a user-friendly format
def show(headline: str, preds: list):
    # Get the best topic and its confidence score
    top_label, top_score = best_topic(preds)
    # Print a separator line
    print("\n" + "=" * 60)
    # Print the classifier title
    print("📰 News Topic Classifier")
    # Print another separator line
    print("=" * 60)
    # Print the original headline
    print("Headline:", headline)
    # Print the best predicted topic
    print(f"Best topic: {top_label}")
    # Print the confidence of the best topic with its bar visualization
    print(f"Confidence: {round(top_score*100,1)}% [{bar(top_score)}]")

    # Print a new line and a heading for top 3 guesses
    print("\nTop 3 guesses:")
    # Sort predictions by score in descending order and take the top 3
    top3 = sorted(preds, key=lambda x: x["score"], reverse=True)[:3]
    # Iterate through the top 3 predictions and print them
    for i, p in enumerate(top3, start=1):
        print(f"{i}. {p['label']:<11} {round(p['score']*100,1)}% [{bar(p['score'])}]")
    # Print a final separator line
    print("=" * 60)


# Define the main function to run the interactive news topic classifier
def main():
    # Welcome message and instructions
    print("Welcome! Type a news headline and I'll guess the topic.")
    # Display the available topics
    print("Topics:", ", ".join(TOPICS))
    # Instruction to exit the program
    print("Type 'exit' to stop.\n")

    # Start an infinite loop to continuously ask for headlines
    while True:
        # Get headline input from the user and remove leading/trailing whitespace
        headline = input("Headline: ").strip()
        # Check if the user wants to exit
        if headline.lower() == "exit":
            print("Bye! Keep coding 👋") # Exit message
            break # Break out of the loop
        # Check if the headline input is empty
        if not headline:
            print("Please type a headline (not empty).\n") # Prompt for non-empty headline
            continue # Continue to the next iteration of the loop
        try:
            # Ask Hugging Face API to classify the headline
            preds = ask_hf(headline)
            # Validate the API response structure
            if isinstance(preds, list) and preds and "label" in preds[0]:
                show(headline, preds) # Display the results
            else:
                print("Oops! Unexpected reply:", preds) # Handle unexpected API response
        except Exception as e:
            # Catch any exceptions during API call or processing
            print("\n⚠️ Oops! Something went wrong.")
            print("Reason:", e) # Print the error reason
            print("Tip: Check HF_API_KEY + internet.\n") # Provide a tip for debugging


# Check if the script is being run directly (not imported as a module)
if __name__ == "__main__":
    main() # Call the main function to start the application

In [ ]:
import requests

# Add your Hugging Face API token here
API_TOKEN = "YOUR_HUGGING_FACE_API_TOKEN"

# Hugging Face spam classification model
API_URL = "https://api-inference.huggingface.co/models/mrm8488/bert-tiny-finetuned-sms-spam-detection"

headers = {
    "Authorization": f"Bearer {API_TOKEN}"
}

def classify_message(message):
    payload = {
        "inputs": message
    }

    response = requests.post(API_URL, headers=headers, json=payload)

    if response.status_code != 200:
        return "Error: Unable to classify message"

    result = response.json()

    label = result[0][0]["label"]
    score = result[0][0]["score"]

    if label.lower() == "spam":
        return f"Spam Message detected with confidence: {score:.2f}"
    else:
        return f"Safe Message detected with confidence: {score:.2f}"


# Main program
print("Spam vs Safe Message Classifier")
print("--------------------------------")

while True:
    user_message = input("\nEnter a message to classify or type 'exit' to quit: ")

    if user_message.lower() == "exit":
        print("Program ended.")
        break

    prediction = classify_message(user_message)
    print(prediction)